## Anna Giczewska ST544 - Final Project ¶
Date 5/2/2026

Short Project Description:

In [1]:
import pandas as pd
from pyspark.sql import SparkSession

from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, Binarizer
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml.feature import VectorAssembler, PCA

from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col

In [2]:
spark = SparkSession.builder.appName("final_project").getOrCreate()
spark.sparkContext.setLogLevel("WARN")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 06:54:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# read the modeling data with pandas first
power_pd = pd.read_csv("power_ml_data.csv")

print(power_pd.shape)
power_pd.head()

(47174, 10)


,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


In [4]:
# convert pandas data frame to spark data frame
power_df = spark.createDataFrame(power_pd)

power_df.printSchema()
power_df.show(5)

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

In [5]:
# just to see the columns clearly
print(power_df.columns)

['Temperature', 'Humidity', 'Wind_Speed', 'General_Diffuse_Flows', 'Diffuse_Flows', 'Power_Zone_1', 'Power_Zone_2', 'Power_Zone_3', 'Month', 'Hour']


In [6]:
# cast Hour to double and create label
sql_stage = SQLTransformer(
    statement="""
    SELECT *,
           CAST(Hour AS DOUBLE) AS Hour_double,
           Power_Zone_3 AS label
    FROM __THIS__
    """
)

# binary hour variable
hour_bin = Binarizer(
    threshold=6.5,
    inputCol="Hour_double",
    outputCol="Hour_binary"
)

# one-hot encode Month
month_indexer = StringIndexer(
    inputCol="Month",
    outputCol="Month_index",
    handleInvalid="keep"
)

month_encoder = OneHotEncoder(
    inputCols=["Month_index"],
    outputCols=["Month_vec"]
)

In [7]:
# put weather variables together for PCA
weather_assembler = VectorAssembler(
    inputCols=[
        "Temperature",
        "Humidity",
        "Wind_Speed",
        "General_Diffuse_Flows",
        "Diffuse_Flows"
    ],
    outputCol="weather_features"
)

# PCA with 2 components
pca_stage = PCA(
    k=2,
    inputCol="weather_features",
    outputCol="pca_features"
)

In [8]:
# final feature vector
final_assembler = VectorAssembler(
    inputCols=[
        "pca_features",
        "Hour_binary",
        "Power_Zone_1",
        "Power_Zone_2",
        "Month_vec"
    ],
    outputCol="features"
)

feature_pipeline = Pipeline(stages=[
    sql_stage,
    hour_bin,
    month_indexer,
    month_encoder,
    weather_assembler,
    pca_stage,
    final_assembler
])

feature_model = feature_pipeline.fit(power_df)
feature_df = feature_model.transform(power_df)

feature_df.select("label", "features").show(5, truncate=False)

+-----------+-------------------------------------------------------------------------------------+
|label      |features                                                                             |
+-----------+-------------------------------------------------------------------------------------+
|20240.96386|(17,[0,1,3,4,8],[1.7944048636569547,-0.7412746447404446,34055.6962,16128.87538,1.0]) |
|20131.08434|(17,[0,1,3,4,8],[1.8060408300982318,-0.710853423955847,29814.68354,19375.07599,1.0]) |
|19668.43373|(17,[0,1,3,4,8],[1.8102297630563908,-0.7283113191158926,29128.10127,19006.68693,1.0])|
|18899.27711|(17,[0,1,3,4,8],[1.7986676517408848,-0.7220913032199933,28228.86076,18361.09422,1.0])|
|18442.40964|(17,[0,1,3,4,8],[1.8632872016379716,-0.7323046647776549,27335.6962,17872.34043,1.0]) |
+-----------+-------------------------------------------------------------------------------------+
only showing top 5 rows


In [9]:
#set up model
lr = LinearRegression(
    featuresCol="features",
    labelCol="label"
)

full_pipeline = Pipeline(stages=[
    sql_stage,
    hour_bin,
    month_indexer,
    month_encoder,
    weather_assembler,
    pca_stage,
    final_assembler,
    lr
])

In [10]:
reg_values = [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]
enet_values = [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]

param_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, reg_values)
    .addGrid(lr.elasticNetParam, enet_values)
    .build()
)

In [11]:
rmse_eval = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

cv = CrossValidator(
    estimator=full_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=rmse_eval,
    numFolds=5
)

In [12]:
cv_model = cv.fit(power_df)

best_model = cv_model.bestModel
best_lr = best_model.stages[-1]

print("best regParam:", best_lr.getRegParam())
print("best elasticNetParam:", best_lr.getElasticNetParam())
print("best CV RMSE:", min(cv_model.avgMetrics))

26/05/03 06:54:36 WARN Instrumentation: [7b3fa445] regParam is zero, which might cause numerical instability and overfitting.
26/05/03 06:54:36 WARN Instrumentation: [7b3fa445] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/05/03 06:54:38 WARN Instrumentation: [40f5c700] regParam is zero, which might cause numerical instability and overfitting.
26/05/03 06:54:40 WARN Instrumentation: [74e82951] regParam is zero, which might cause numerical instability and overfitting.
26/05/03 06:54:40 WARN Instrumentation: [74e82951] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/05/03 06:54:41 WARN Instrumentation: [d1e22466] regParam is zero, which might cause numerical instability and overfitting.
26/05/03 06:54:42 WARN Instrumentation: [b222a665] regParam is zero, which might cause numerical instability and overfitting.
26/05/03 06:54:43 WARN Instrumentation: [0070d274] regParam is zero, which might c

best regParam: 0.75
best elasticNetParam: 0.75
best CV RMSE: 2147.986668015081


In [13]:
train_pred = best_model.transform(power_df)

train_rmse = rmse_eval.evaluate(train_pred)
print("training RMSE:", train_rmse)

training RMSE: 2147.099438010557


In [14]:
resid_df = (
    train_pred
    .withColumn("residual", col("label") - col("prediction"))
    .select("label", "prediction", "residual")
)

resid_df.show(20, truncate=False)

+-----------+------------------+------------------+
|label      |prediction        |residual          |
+-----------+------------------+------------------+
|20240.96386|20876.45587318845 |-635.4920131884501|
|20131.08434|18654.62940223957 |1476.4549377604308|
|19668.43373|18199.288795695953|1469.1449343040476|
|18899.27711|17585.480347244946|1313.7967627550534|
|18442.40964|16992.31576187153 |1450.0938781284713|
|18130.12048|16512.89423127975 |1617.2262487202497|
|17945.06024|16088.64000186192 |1856.4202381380783|
|17459.27711|15718.223248833125|1741.0538611668744|
|17025.54217|15266.757811671789|1758.784358328212 |
|16794.21687|14934.17540893735 |1860.0414610626503|
|16638.07229|14648.502989827888|1989.569300172112 |
|16395.18072|14411.113349504813|1984.0673704951878|
|16117.59036|14079.027877296838|2038.5624827031625|
|15822.6506 |13621.224254866049|2201.426345133952 |
|15672.28916|13446.847028039647|2225.4421319603534|
|15597.10843|13298.947332188829|2298.161097811171 |
|15510.36145

In [15]:
import os

os.makedirs("stream_data", exist_ok=True)
os.makedirs("checkpoint_folder", exist_ok=True)

In [16]:
stream_schema = power_df.schema
stream_schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', LongType(), True), StructField('Hour', LongType(), True)])

In [17]:
stream_df = (
    spark.readStream
    .option("header", True)
    .schema(stream_schema)
    .csv("stream_data")
)

print(stream_df.isStreaming)

True


In [18]:
stream_predictions = (
    best_model
    .transform(stream_df)
    .withColumn("residual", col("label") - col("prediction"))
    .select("label", "prediction", "residual")
)

In [19]:
query = (
    stream_predictions.writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", False)
    .option("checkpointLocation", "checkpoint_folder")
    .start()
)

26/05/03 07:21:41 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+-------------------+
|label      |prediction        |residual           |
+-----------+------------------+-------------------+
|16498.49246|19023.842292447298|-2525.3498324472966|
|17140.64516|18325.5781475501  |-1184.932987550099 |
|10022.16867|10553.202280834445|-531.0336108344454 |
|15307.95181|14163.173873438149|1144.7779365618517 |
|9254.261705|7058.959988144543 |2195.301716855458  |
+-----------+------------------+-------------------+

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+
|label      |prediction        |residual           |
+-----------+------------------+-------------------+
|15268.66397|17766.343529653055|-2497.6795596530555|
|24261.46734|24361.05272345124 |-99.5853834512418  |
|14808.09717|14533.474201505367|274.622968494632   |
|32403.012